In [1]:
%load_ext autoreload
%autoreload 2

### Rolling origin backtests with selected features:

* Repeat the rolling origin backtesting experiments (particularly rolling window) for the selected features on full items or selected items (for faster computation)

In [2]:
import sys
from pathlib import Path

# Adds the root_dir (parent of notebooks/) to sys.path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)


BASE_DIR = Path.cwd().parent
BASE_DIR

DATA_DIR = BASE_DIR/"data"/"processed"
MODELS_DIR = BASE_DIR /"models"
parent_dir

'/media/ashfaque/datas/ML-projects/retail-forecast-system'

In [3]:
# all libraries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from datetime import datetime
import joblib
import lightgbm as lgb 
plt.style.use('seaborn-v0_8-darkgrid')

from src.metrics import wrmsse, calculate_mape, calculate_wape, bias
from src.features import GetLagRollFeatures , get_avg_sales, get_price_features, get_trend_features
from src.utils import get_items_top, get_items_with_min_history
from src.pipeline import recursive_forecast_batch 
from src.backtests import generate_expanding_windows,generate_rolling_windows, split_data
from train_and_eval import train_models


import gc 
import warnings

warnings.filterwarnings(action="ignore")

In [4]:
# load the data

sales_ca1 = pd.read_parquet(DATA_DIR/"sales_known_ca_1.parquet")
sales_future_ca1 = pd.read_parquet(DATA_DIR/"sales_future_ca_1.parquet")

selected_features = pd.read_pickle(MODELS_DIR/"feature_cols_recursive_ca1.pkl")

print(sales_ca1.shape, sales_future_ca1.shape)
print(" number of training days: ",(sales_ca1['date'].max()-sales_ca1['date'].min()).days)
print(" number of future unknown days: ", (sales_future_ca1['date'].max()-sales_future_ca1['date'].min()).days)
print(" maximum training date: ", sales_ca1['date'].max())
print(" number of unique items in store: ",len(sales_ca1['item_id'].unique()))

(4367553, 25) (420714, 25)
 number of training days:  1802
 number of future unknown days:  137
 maximum training date:  2016-01-05 00:00:00
 number of unique items in store:  3047


##  Backtesting experiments:

* model = lightgbm
* forecasting method = recursive
* horizon = 28
* history = full
* metric  = wrmsse 


### Walk forward Rolling window

In [5]:
# generate rolling windows for the entire dataset
rolling_windows = generate_rolling_windows(sales_ca1,training_window=730,horizon=28,
                                           step_size=120,date_col='date')

rolling_windows

[{'window_id': 'window_1',
  'train_start': Timestamp('2011-01-29 00:00:00'),
  'train_end': Timestamp('2013-01-28 00:00:00'),
  'test_start': Timestamp('2013-01-28 00:00:00'),
  'test_end': Timestamp('2013-02-25 00:00:00')},
 {'window_id': 'window_2',
  'train_start': Timestamp('2011-05-29 00:00:00'),
  'train_end': Timestamp('2013-05-28 00:00:00'),
  'test_start': Timestamp('2013-05-28 00:00:00'),
  'test_end': Timestamp('2013-06-25 00:00:00')},
 {'window_id': 'window_3',
  'train_start': Timestamp('2011-09-26 00:00:00'),
  'train_end': Timestamp('2013-09-25 00:00:00'),
  'test_start': Timestamp('2013-09-25 00:00:00'),
  'test_end': Timestamp('2013-10-23 00:00:00')},
 {'window_id': 'window_4',
  'train_start': Timestamp('2012-01-24 00:00:00'),
  'train_end': Timestamp('2014-01-23 00:00:00'),
  'test_start': Timestamp('2014-01-23 00:00:00'),
  'test_end': Timestamp('2014-02-20 00:00:00')},
 {'window_id': 'window_5',
  'train_start': Timestamp('2012-05-23 00:00:00'),
  'train_end': Tim

* between each window there is atleast 4 month gap. So we choose only 3 or 4 windows for faster calculations, window_1, window_4, window_7, window_9 would be good


In [29]:
full_df = sales_ca1.copy()


for col in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    if col in full_df.columns:
        full_df[col] = (
            full_df[col].astype(str).replace("nan", "None")
        )
        full_df[col] = full_df[col].astype("category")

In [8]:
lag_and_roll = [feat for feat in selected_features if 'lag' in feat or 'rolling' in feat]
lag_and_roll

['rolling_lag_28_win_7',
 'rolling_lag_28_win_28',
 'rolling_max_60',
 'rolling_max_28',
 'rolling_mean_90',
 'rolling_mean_7',
 'lag_60',
 'rolling_max_90',
 'rolling_max_7',
 'lag_28',
 'lag_7',
 'rolling_mean_60',
 'lag_90',
 'rolling_mean_28']

In [ ]:
get_lags_rolls = GetLagRollFeatures()


In [ ]:
# remove the duplicates in selected features and save it again 
feats_unique = list(set(selected_features))
#overwrite the feature col

# feature_file = MODELS_DIR/ "feature_cols_recursive_ca1.pkl"
# with open(feature_file,'wb') as f:
#     joblib.dump(feats_unique,f)

# print("sucessfully overwrote unique features")

sucessfully overwrote unique features


In [78]:
window_ids  = set(['window_1','window_4','window_7','window_9'])

wrmsse_results = {}
bias_results = {}


for idx,window in enumerate(rolling_windows):
    if window['window_id'] in window_ids:
        window_start = window['train_start'] 
        window_end = window['train_end']
        print(f"Rolling Window from {window_start} to {window_end}\n-------------------------------------------------------")
        # split the data
        train_wndow, test_wndow = split_data(full_df,window_start,window_end)
        # filter to items which have min of 100 days
        full_window = pd.concat((train_wndow,test_wndow),ignore_index=True)
        filter_window = get_items_with_min_history(full_window,min_history_days=130)
        
        # calculate the features for full window length: 
        filter_window = get_price_features(filter_window) # price features
        filter_window = get_lags_rolls.add_lags(filter_window,lags=[1,7, 28,60,90])
        filter_window = get_lags_rolls.add_rolling_mean(filter_window,windows=[7,28,60,90])
        filter_window = get_lags_rolls.add_rolling_max(filter_window,windows=[7,28,60,90])
        filter_window = get_lags_rolls.add_rolling_on_lag(filter_window,lags=[28],windows=[7,28])
        filter_window = get_trend_features(filter_window)

        # split the data again to train and test, now both contain price features
        train, test = split_data(filter_window,window_start,window_end)
        eval_df = test.copy()
        # now drop the lag and roll features from test data
        test = test.drop(columns = lag_and_roll+['lag_1','selling_trend','demand_vs_historical_mean'])

        # print(list(train.columns),list(test.columns))
        # print("Duplicate columns in train:", train.columns[train.columns.duplicated()].tolist())
        # print("Duplicate columns in test :", test.columns[test.columns.duplicated()].tolist())

        # now train the data 
        window_models, cat_categories, _ = train_models(train,eval_df,feature_cols=feats_unique)
        # recursive predict 
        pred_window, items_df = recursive_forecast_batch(models= window_models,history_df=train,
                                                         future_static_df=test,feature_cols=feats_unique,cat_categories=cat_categories,
                                                         lags=[1,7,28,60,90],rolling_mean_windows=[7,28,60,90],
                                                         rolling_max_windows=[7,28,60,90],id_col='item_id',max_lookback_days=100)
        score_wrmsse = wrmsse(train,test,pred_window)
   
        wape = calculate_wape(test['sales'],pred_window['sales_pred'])
        bias_score = bias(test['sales'],pred_window['sales_pred'])

        window_id = window['window_id']
        wrmsse_results[window_id] = score_wrmsse
        bias_results[window_id] = bias_score
        
        # print the values 
        print(f'\n WRMSSE: {score_wrmsse} | WAPE: {wape} | bias: {bias_score}\n')



Rolling Window from 2011-01-29 00:00:00 to 2013-01-28 00:00:00
-------------------------------------------------------
total history:  758
Total unique items: 2325
Items with full history: 2172

 WRMSSE: 0.8293899358097719 | WAPE: 1.3720763329316745 | bias: -0.0833626675043914

Rolling Window from 2012-01-24 00:00:00 to 2014-01-23 00:00:00
-------------------------------------------------------
total history:  758
Total unique items: 2700
Items with full history: 2598

 WRMSSE: 0.8208277509172486 | WAPE: 1.5215789539657496 | bias: 0.047175640054956273

Rolling Window from 2013-01-18 00:00:00 to 2015-01-18 00:00:00
-------------------------------------------------------
total history:  758
Total unique items: 3019
Items with full history: 2945

 WRMSSE: 0.8142734562383183 | WAPE: 1.4873216940887175 | bias: 0.025072818160656266

Rolling Window from 2013-09-15 00:00:00 to 2015-09-15 00:00:00
-------------------------------------------------------
total history:  758
Total unique items: 30

In [87]:
# mean and std of wrmsse and bias across windows:
wrmsse_vals = list(wrmsse_results.values())
bias_vals  = list(bias_results.values())

mean_wrmsse,wrmsse_std = np.array(wrmsse_vals).mean(),np.array(wrmsse_vals).std() 
mean_bias, std_bias = np.array(bias_vals).mean(),np.array(bias_vals).std()

print(f"WRMSSE: mean: {mean_wrmsse}| std: {wrmsse_std}\n BIAS: mean: {mean_bias} | std: {std_bias}")

WRMSSE: mean: 0.8214685438035086| std: 0.005360402600009102
 BIAS: mean: 0.006486325898519015 | std: 0.052461031630695036


While a single static window gave a slightly lower WRMSSE (0.8178), static benchmarks suffer from survivor bias by training on items introduced far into the future. 
* Rolling window metrics (WRMSSE) mean of 0.8214 and a 0.005 standard deviation, with average bias ~0.006) show that the pipeline generalizes reliably across changing time horizons and expanding catalog sizes.
* Summary of Backtest Performance across WindowsTemporal Generalization: The low spread in WRMSSE ($\text{std} \approx 0.006$) demonstrates that model performance does not degrade as retail sales regimes shift between 2011 and 2015.
* Realistic Catalog Dynamics: Evaluating only active items per window removes lookahead bias and simulates real-world production conditions.
* Well-Calibrated Bias: An average bias of $\approx +0.006$ across windows indicates that, Tweedie objective function maintains solid calibration without systemic over- or under-forecasting.

In [108]:
window_models['point'].best_iteration_

155

* we can use this n_estimators in our final model. Since this values is from one window, we can use n_estimators=200 as a safety net.

In [79]:
def plot(raw_data, pred_data,items):
    fig, ax = plt.subplots(nrows=len(items),figsize=(15,10))

    for i, item in enumerate(items):
        raw = raw_data[raw_data['item_id']==item]
        pred = pred_data[pred_data['item_id']==item]

        ax[i].plot(raw['date'],raw['sales'],label='actual')
        ax[i].plot(pred['date'],pred['sales_pred'],label='predicted')
        ax[i].fill_between(pred['date'],pred['q10'],pred['q90'],alpha=0.2,color='green')
        
        ax[i].legend()
        ax[i].set_ylabel(item)
    plt.show()
    

**the best time period for our calculation is training on 2 years of data**

## benchmarking with a baseline: seasonal naive

In [ ]:
# Create baseline predictions: sales from 28 days ago

full_data_2yr = full_df.copy()

two_year_cutoff = full_data_2yr['date'].max()-pd.Timedelta(days=730)
final_train= full_data_2yr[full_data_2yr['date']>=two_year_cutoff]

cutoff = final_train['date'].max()- pd.Timedelta(days=28) 

train = final_train[final_train['date']<=cutoff] 
test  = final_train[final_train['date']>cutoff]


train_sorted =  train .sort_values(['item_id','date'])

pred_baseline = train_filtered[['item_id','date','sales']].copy()
pred_baseline['date'] = pred_baseline['date'] + pd.Timedelta(days=28)
pred_baseline = pred_baseline.rename(columns={'sales': 'sales_pred'})
pred_baseline['q90'] = pred_baseline['sales_pred']  # no safety margin
pred_baseline['q10'] = pred_baseline['sales_pred']

# Filter to only match test dates
pred_baseline = pred_baseline[pred_baseline['date'].isin(test['date'])]


score = wrmsse(train_sorted,test,pred_baseline)
print(f"wrmsse baseline: {score}")


# # Now use your existing function
# cost_baseline = cost_per_item(test, pred_baseline)
# cost_optimized = cost_per_item(test, pred_df)  # your quantile preds

# # Compare
# comparison = cost_optimized.merge(cost_baseline, on='item_id', suffixes=('_optimized','_baseline'))
# comparison['savings_pct'] = ((comparison['total_cost_baseline'] - comparison['total_cost_optimized']) / comparison['total_cost_baseline'] * 100).round(2)

# total_savings_pct = ((comparison['total_cost_baseline'].sum() - comparison['total_cost_optimized'].sum()) / comparison['total_cost_baseline'].sum() * 100)
# print(f"Overall savings with lgb model compared to baseline: {total_savings_pct:.1f}%")

total history:  1802
Total unique items: 3047
Items with full history: 3043


KeyError: "['FOODS_2_117', 'FOODS_2_209', 'FOODS_2_248', 'FOODS_2_379', 'FOODS_3_264', 'FOODS_3_350', 'HOBBIES_2_132', 'HOUSEHOLD_1_441'] not in index"

# Final Model
* From the rolling window walk forward validation set, we observed that a training period of 2 years have better score 
* Lower variance — more stable across different time periods (std 0.036)
* Better average — mean 0.809 was the best

In [115]:
# select the last two years of data from 

full_data_2yr = full_df.copy()
two_year_cutoff = full_data_2yr['date'].max()-pd.Timedelta(days=730)
final_train= full_data_2yr[full_data_2yr['date']>=two_year_cutoff]

# filter the dataset to items with atleast 100 days (since we have lag_90 in the featureset)

final_train_filtered = get_items_with_min_history(final_train,min_history_days=100)
items_unique = final_train_filtered['item_id'].nunique()
min_date = final_train['date'].min()
max_date = final_train['date'].max()

print("after cutoff: ",min_date,max_date,(max_date-min_date).days, ' days')

total history:  730
Total unique items: 3047
Items with min 100 days: 3043
after cutoff:  2014-01-05 00:00:00 2016-01-05 00:00:00 730  days


In [116]:

# get the lag, roll, price , trend features on full data

final_train_filtered = get_price_features(final_train_filtered)
final_train_filtered = get_lags_rolls.add_lags(final_train_filtered,lags=[1,7,28,60,90])
final_train_filtered = get_lags_rolls.add_rolling_mean(final_train_filtered,windows=[7,28,60,90])
final_train_filtered = get_lags_rolls.add_rolling_max(final_train_filtered,windows=[7,28,60,90])
final_train_filtered = get_lags_rolls.add_rolling_on_lag(final_train_filtered,lags=[28],windows=[7,28])
final_train_filtered = get_trend_features(final_train_filtered)

# assert that final train has all the features from selected features


In [ ]:
# train 

final_models, cat_categories,params = train_models(final_train_filtered,test_df=None,quantiles=True,feature_cols= feats_unique,
                                                   n_estimators=200)


quantile models training.


In [119]:
import pickle

model_dir = BASE_DIR/'models'

point_model = final_models['point']
model_q10  = final_models['q10']
model_q90  = final_models['q90']

# unique items in the model
unique_items = final_train_filtered['item_id'].unique().tolist()
filename_output = model_dir / 'items_min_100days_ca1.pkl'

with open(filename_output, 'wb') as f:
       pickle.dump({'items': unique_items}, f)

print(f"saved items name with full history")

joblib.dump(point_model,model_dir/'lgb_ca1_2yr_point.pkl')
joblib.dump(model_q10,model_dir/'lgb_ca1_2yr_q10.pkl')
joblib.dump(model_q90, model_dir/'lgb_ca1_2yr_q90.pkl')


# save this model for future

saved items name with full history


['/media/ashfaque/datas/ML-projects/retail-forecast-system/models/lgb_ca1_2yr_q90.pkl']

In [120]:
# save the training data too 
data_file = DATA_DIR/ 'final_training_ca1.parquet'

final_train_filtered.sort_values(['item_id','date']).to_parquet(data_file,index=False)